# Salidas reales de la última corrida CUDA

**RUN:** `RUN-20261117T234658Z-gpu-05` · **EXP-003**. El entrenamiento ya fue ejecutado y sus artefactos
están sellados. Las celdas siguientes **leen y muestran las salidas originales**:
no simulan un entrenamiento ni vuelven a entrenar. Se conservan abajo el código
original protegido y la implementación CUDA documentada.


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/Proyecto 3 CUDA.pdf").is_file())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
EJECUTAR_ORIGINAL = False
EJECUTAR_GPU = False  # Esta vista solo recupera evidencia real; no entrena.
EXPERIMENTO = "EXP-003"
print("Intérprete:", sys.executable)
print("CPU original habilitada:", EJECUTAR_ORIGINAL, "GPU habilitada:", EJECUTAR_GPU)


In [ ]:
import csv, hashlib
import pandas as pd
from IPython.display import display, Image
ULTIMO_RUN = ROOT / "resultados/paralelo/EXP-003/RUN-20261117T234658Z-gpu-05"
salida_real = json.loads((ULTIMO_RUN / "salida.json").read_text(encoding="utf8"))
assert salida_real["codigo"] == 0
print("RUN real:", ULTIMO_RUN.name)
print("Código de salida del entrenamiento:", salida_real["codigo"])
print("No se ejecuta fit ni se crean corridas nuevas en esta visualización.")


## 1. Salida original de las 15 épocas

Texto recuperado literalmente de `stdout.log` del RUN indicado.

In [ ]:
log_real = (ULTIMO_RUN / "stdout.log").read_text(encoding="utf8")
lineas_epoca = [linea for linea in log_real.splitlines() if linea.startswith("EPOCH ")]
assert len(lineas_epoca) == 15
for linea in lineas_epoca:
    print(linea)


## 2. Resultados por época

Datos reales de `epocas.csv`. El tiempo de época incluye su validación CPU.

In [ ]:
epocas_reales = pd.read_csv(ULTIMO_RUN / "epocas.csv")
assert len(epocas_reales) == 15 and int(epocas_reales.iloc[-1]["pasos_acumulados"]) == 1440
display(epocas_reales)


## 3. Métricas finales de prueba y validación

In [ ]:
metricas_reales = json.loads((ULTIMO_RUN / "metricas.json").read_text(encoding="utf8"))
print("Épocas:", metricas_reales["epocas"])
print("Actualizaciones reales:", metricas_reales["pasos"])
print(f"Test accuracy: {metricas_reales['test']['accuracy']:.8%}")
print(f"Test loss: {metricas_reales['test']['loss']:.10f}")
print("Última validación CPU:")
print(json.dumps(metricas_reales["val_final"], indent=2, ensure_ascii=False))


## 4. Tiempos reales, cómputo y transferencias

Son mediciones de la última corrida. Los intervalos de eventos y la validación están anidados en el entrenamiento: no deben sumarse otra vez al total.

In [ ]:
tiempos_reales = pd.read_csv(ULTIMO_RUN / "tiempos.csv")
display(tiempos_reales[["fase", "tipo_medicion", "duracion_s", "epocas", "pasos", "motivo"]])
tiempo_train = float(tiempos_reales.loc[tiempos_reales.fase == "entrenamiento_CNN-B", "duracion_s"].iloc[0])
print(f"Entrenamiento completo de esta corrida: {tiempo_train:.6f} s ({tiempo_train/60:.4f} min)")
print("Incluye H2D/D2H, staging y validación CPU; no es tiempo exclusivo de kernels.")


## 5. Curvas de la última corrida, obtenidas de sus datos originales

In [ ]:
import io
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epocas_reales.epoca, epocas_reales.train_accuracy, label="Entrenamiento")
axes[0].plot(epocas_reales.epoca, epocas_reales.val_accuracy, label="Validación CPU")
axes[0].set(xlabel="Época", ylabel="Accuracy", ylim=(0, 1.02))
axes[1].plot(epocas_reales.epoca, epocas_reales.train_loss, label="Entrenamiento")
axes[1].plot(epocas_reales.epoca, epocas_reales.val_loss, label="Validación CPU")
axes[1].set(xlabel="Época", ylabel="Pérdida crossentropy")
for ax in axes:
    ax.legend(); ax.grid(alpha=.15)
fig.suptitle(ULTIMO_RUN.name)
fig.tight_layout()
buffer = io.BytesIO()
fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
display(Image(data=buffer.getvalue()))
plt.close(fig)


## 6. Hardware, estado guardado y procedencia verificable

In [ ]:
hardware_real = json.loads((ULTIMO_RUN / "hardware.json").read_text(encoding="utf8"))
print("GPU:", hardware_real["device"]["name"])
print("Compute capability:", hardware_real["device"]["compute_capability"])
print("cuBLAS:", hardware_real["cublas_version"])
print("Buffers propios (MiB):", hardware_real["buffers_bytes"] / 2**20)
modelo_real = json.loads((ULTIMO_RUN / "modelo-guardado.json").read_text(encoding="utf8"))
print(json.dumps(modelo_real, indent=2, ensure_ascii=False))
proveniencia = pd.DataFrame([{"archivo": nombre, "sha256": hashlib.sha256((ULTIMO_RUN/nombre).read_bytes()).hexdigest()}
    for nombre in ["stdout.log", "epocas.csv", "metricas.json", "tiempos.csv", "hardware.json", "modelo-guardado.json"]])
display(proveniencia)


## A. Código original copiado (referencia, ejecución opcional)

Los textos siguientes pertenecen al laboratorio original. Sus conclusiones no son resultados CUDA nuevos.

# Laboratorio 3 Deep Learning: Reconocimiento de Lenguaje de Señas (ASL)

SignBridge, una startup guatemalteca de tecnología inclusiva, nos contrató para construir el primer prototipo de un motor de reconocimiento de letras del alfabeto de Lenguaje de Señas Americano (ASL) a partir de imágenes de manos.

## 0. Configuración e imports

In [ ]:
if EJECUTAR_ORIGINAL:
    import os
    import json
    import random
    import glob
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from PIL import Image
    from sklearn.model_selection import train_test_split

    import kagglehub

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)

    sns.set_theme(style="whitegrid")
    plt.rcParams["figure.figsize"] = (10, 6)

## 1. Descarga del dataset

Usamos `kagglehub` para descargar el dataset [ASL Alphabet](https://www.kaggle.com/datasets/grassknoted/asl-alphabet). `kagglehub` cachea el dataset localmente (fuera del repositorio, típicamente en `~/.cache/kagglehub`) y solo vuelve a descargarlo si no existe, por lo que es seguro volver a correr esta celda. El dataset **no se sube al repositorio** (ver `.gitignore`); cualquier persona que clone el repo simplemente vuelve a correr esta celda para obtenerlo.

In [ ]:
if EJECUTAR_ORIGINAL:
    # Download latest version
    path = kagglehub.dataset_download("grassknoted/asl-alphabet")

    print("Path to dataset files:", path)

    DATASET_DIR = Path(path)
    TRAIN_DIR = DATASET_DIR / "asl_alphabet_train" / "asl_alphabet_train"
    TEST_DIR = DATASET_DIR / "asl_alphabet_test" / "asl_alphabet_test"

    assert TRAIN_DIR.exists(), f"No se encontró la carpeta de entrenamiento en {TRAIN_DIR}"
    assert TEST_DIR.exists(), f"No se encontró la carpeta de prueba oficial en {TEST_DIR}"

    CLASSES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
    print(f"Número de clases: {len(CLASSES)}")
    print(CLASSES)

## 2. Estructura general del dataset

Contamos cuántas imágenes hay por clase dentro de la carpeta de entrenamiento y revisamos qué tan grande es el set de prueba oficial de Kaggle.

In [ ]:
if EJECUTAR_ORIGINAL:
    counts = {c: len(list((TRAIN_DIR / c).glob("*.jpg"))) for c in CLASSES}
    df_counts = pd.DataFrame({"clase": list(counts.keys()), "n_imagenes": list(counts.values())})
    df_counts = df_counts.sort_values("clase").reset_index(drop=True)

    test_files = list(TEST_DIR.glob("*.jpg"))

    print(f"Total de imágenes de entrenamiento disponibles: {df_counts['n_imagenes'].sum()}")
    print(f"Número de clases: {df_counts.shape[0]}")
    print(f"Imágenes en el set de prueba OFICIAL de Kaggle: {len(test_files)}")
    df_counts

**Hallazgo:** el dataset trae 29 clases (A-Z, `space`, `del`, `nothing`) con ~3,000 imágenes cada una (~87,000 en total), es decir, está **perfectamente balanceado por clase**. El set de prueba oficial de Kaggle trae apenas ~28 imágenes (una por letra), a propósito, para forzar a construir nuestro propio conjunto de entrenamiento/validación/prueba a partir de las ~87,000 imágenes de entrenamiento (ver sección 2.4 del EDA).

## Ejercicio 1: Ejemplos de letras y variabilidad intra-clase

Mostramos varias muestras de distintas letras, incluyendo tanto letras "fáciles" como letras que visualmente podrían confundirse entre sí, para observar la variabilidad dentro de una misma clase (rotación de la mano, encuadre, iluminación, etc.).

In [ ]:
if EJECUTAR_ORIGINAL:
    def show_class_samples(classes, n_samples=5, seed=SEED):
        rng = random.Random(seed)
        fig, axes = plt.subplots(len(classes), n_samples, figsize=(2.2 * n_samples, 2.2 * len(classes)))
        for i, c in enumerate(classes):
            files = list((TRAIN_DIR / c).glob("*.jpg"))
            sample_files = rng.sample(files, n_samples)
            for j, f in enumerate(sample_files):
                img = Image.open(f)
                ax = axes[i, j] if len(classes) > 1 else axes[j]
                ax.imshow(img)
                ax.axis("off")
                if j == 0:
                    ax.set_ylabel(c, fontsize=12)
                    ax.text(-0.15, 0.5, c, fontsize=14, fontweight="bold",
                            transform=ax.transAxes, va="center", ha="right")
        plt.suptitle("Variabilidad intra-clase: 5 muestras por letra", fontsize=14, y=1.0)
        plt.tight_layout()
        plt.show()

    show_class_samples(["A", "B", "L", "M", "N", "S", "space"], n_samples=5)

**Hallazgo:** dentro de una misma clase la seña mantiene la misma configuración de dedos, pero varía ligeramente la rotación/inclinación de la mano, el encuadre respecto a la cámara y, en menor medida, la iluminación. El fondo es consistente entre imágenes de la misma sesión de captura, lo cual es importante para la reflexión de accesibilidad más adelante (Ejercicio 9).

## Ejercicio 2: Análisis exploratorio de los datos

### 2.1 Resolución y formato

In [ ]:
if EJECUTAR_ORIGINAL:
    rng = random.Random(SEED)
    sample_files = []
    for c in rng.sample(CLASSES, 10):
        files = list((TRAIN_DIR / c).glob("*.jpg"))
        sample_files.extend(rng.sample(files, 5))

    props = []
    for f in sample_files:
        img = Image.open(f)
        props.append({"archivo": f.name, "clase": f.parent.name, "tamaño": img.size,
                      "modo": img.mode, "formato": img.format})

    df_props = pd.DataFrame(props)
    print(df_props["tamaño"].value_counts())
    print(df_props["modo"].value_counts())
    print(df_props["formato"].value_counts())

**Hallazgo:** todas las imágenes muestreadas son archivos `.jpg` (formato JPEG), en modo de color `RGB`, con resolución uniforme de **200x200 píxeles**. No se observan imágenes en escala de grises ni con tamaños distintos, por lo que el dataset es homogéneo en este aspecto y no requiere normalización de formato, solo de resolución y escala de valores (ver preprocesamiento).

### 2.2 Distribución / balance de clases

In [ ]:
if EJECUTAR_ORIGINAL:
    plt.figure(figsize=(14, 6))
    sns.barplot(data=df_counts, x="clase", y="n_imagenes", color="#4C72B0")
    plt.xticks(rotation=90)
    plt.title("Número de imágenes por clase (set de entrenamiento original)")
    plt.ylabel("Número de imágenes")
    plt.xlabel("Clase")
    plt.tight_layout()
    plt.show()

    print(df_counts["n_imagenes"].describe())

**Hallazgo:** el dataset está **balanceado**: las 29 clases tienen prácticamente el mismo número de imágenes (~3,000 cada una, desviación mínima). Esto simplifica el entrenamiento porque no es necesario aplicar técnicas de balanceo de clases (oversampling/undersampling/class weights) al construir nuestros propios conjuntos de entrenamiento/validación/prueba, siempre que mantengamos la proporción al submuestrear.

### 2.3 Letras visualmente similares

Un problema conocido del alfabeto ASL es que varias letras comparten una configuración de mano muy parecida (puño cerrado con variaciones sutiles en la posición del pulgar), lo que las hace fáciles de confundir tanto para personas como para un modelo. Los ejemplos clásicos son **M/N/S** y **U/V/R**.

In [ ]:
if EJECUTAR_ORIGINAL:
    def show_confusable_groups(groups, n_samples=4):
        n_rows = sum(len(g) for g in groups)
        fig, axes = plt.subplots(n_rows, n_samples, figsize=(2.2 * n_samples, 2.2 * n_rows))
        rng = random.Random(SEED)
        row = 0
        for group in groups:
            for c in group:
                files = list((TRAIN_DIR / c).glob("*.jpg"))
                sample_files = rng.sample(files, n_samples)
                for j, f in enumerate(sample_files):
                    img = Image.open(f)
                    ax = axes[row, j]
                    ax.imshow(img)
                    ax.axis("off")
                    if j == 0:
                        ax.text(-0.15, 0.5, c, fontsize=14, fontweight="bold",
                                transform=ax.transAxes, va="center", ha="right")
                row += 1
        plt.suptitle("Grupos de letras visualmente similares", fontsize=14, y=1.0)
        plt.tight_layout()
        plt.show()

    show_confusable_groups([["M", "N", "S"], ["U", "V", "R"]], n_samples=4)

**Hallazgo:** en el grupo **M/N/S** las tres señas son un puño cerrado; la diferencia está únicamente en cuántos dedos cubren el pulgar (M: pulgar bajo tres dedos, N: pulgar bajo dos dedos, S: pulgar sobre los dedos), un detalle muy fino que se pierde fácilmente al reducir la resolución de 200x200 a algo como 64x64. En el grupo **U/V/R** las tres son dos dedos extendidos; la diferencia está en si están juntos (U), separados en "V" (V) o cruzados (R). Estos pares/tríos de letras son los candidatos más probables a generar errores de clasificación y deberán vigilarse explícitamente en la matriz de confusión de los modelos.

### 2.4 Definición del conjunto de entrenamiento / validación / prueba

El set de prueba oficial de Kaggle es demasiado pequeño (~28 imágenes) para evaluar el modelo de forma confiable, así que construimos nuestros propios conjuntos a partir de las ~87,000 imágenes de entrenamiento:

1. **Submuestreo por clase**: por restricciones de cómputo (ver nota del enunciado), tomamos una submuestra aleatoria y estratificada de `SAMPLES_PER_CLASS` imágenes por cada una de las 29 clases (en vez de las ~3,000 disponibles), fijando una semilla para que sea reproducible.
2. **División estratificada 70/15/15**: sobre esa submuestra, dividimos en entrenamiento (70%), validación (15%) y prueba (15%), estratificando por clase con `sklearn.model_selection.train_test_split`, de forma que las proporciones de cada letra se mantengan iguales en los tres conjuntos.
3. El pequeño set de prueba oficial de Kaggle se guarda aparte como una **verificación adicional** (fuera de la métrica principal), útil para revisar cualitativamente el desempeño final del mejor modelo.

Esta estrategia se implementa en la siguiente sección junto con el preprocesamiento, ya que el submuestreo y el preprocesamiento están directamente relacionados.

## Ejercicio 3: Preprocesamiento

**Justificación de la submuestra y la resolución:** el dataset completo (87,000 imágenes a color de 200x200) es demasiado pesado para el tiempo de cómputo disponible en el laboratorio (cargar todo en memoria como arreglos `float32` normalizados ocuparía más de 40 GB de RAM). Siguiendo la recomendación del enunciado, trabajamos con:

- **`SAMPLES_PER_CLASS = 600`** imágenes por clase (≈ 17,400 imágenes en total), suficientes para entrenar redes convolucionales razonablemente bien sin agotar tiempo/memoria.
- **`IMG_SIZE = 64`** (64x64 píxeles) en vez de 200x200, lo que reduce el número de valores por imagen en un factor de ~9.8x (200²/64² ≈ 9.77) manteniendo aún visible la forma general de la mano.
- **Normalización** de los valores de píxel de `[0, 255]` a `[0, 1]` dividiendo entre 255, para estabilizar el entrenamiento de las redes neuronales.

Documentamos cada paso en el código a continuación.

In [ ]:
if EJECUTAR_ORIGINAL:
    SAMPLES_PER_CLASS = 600
    IMG_SIZE = 64

    def build_sampled_file_list(train_dir, classes, samples_per_class, seed=SEED):
        rng = random.Random(seed)
        records = []
        for c in classes:
            files = list((train_dir / c).glob("*.jpg"))
            chosen = rng.sample(files, min(samples_per_class, len(files)))
            for f in chosen:
                records.append({"filepath": str(f), "label": c})
        return pd.DataFrame(records)

    df_sample = build_sampled_file_list(TRAIN_DIR, CLASSES, SAMPLES_PER_CLASS)
    print(f"Total de imágenes submuestreadas: {len(df_sample)}")
    df_sample["label"].value_counts().sort_index()

In [ ]:
if EJECUTAR_ORIGINAL:
    # División estratificada 70/15/15 (train / val / test) sobre la submuestra
    df_train, df_temp = train_test_split(
        df_sample, test_size=0.30, stratify=df_sample["label"], random_state=SEED
    )
    df_val, df_test = train_test_split(
        df_temp, test_size=0.50, stratify=df_temp["label"], random_state=SEED
    )

    print(f"Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}")
    print("\nProporción por clase en train (primeras 5):")
    print(df_train["label"].value_counts(normalize=True).sort_index().head())

In [ ]:
if EJECUTAR_ORIGINAL:
    # Codificación de etiquetas: nombre de clase -> índice entero
    label_to_idx = {c: i for i, c in enumerate(CLASSES)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}

    def load_images(df, img_size=IMG_SIZE):
        X = np.zeros((len(df), img_size, img_size, 3), dtype=np.float32)
        y = np.zeros(len(df), dtype=np.int64)
        for i, row in enumerate(df.itertuples()):
            img = Image.open(row.filepath).convert("RGB").resize((img_size, img_size))
            X[i] = np.asarray(img, dtype=np.float32) / 255.0
            y[i] = label_to_idx[row.label]
        return X, y

    X_train, y_train = load_images(df_train)
    X_val, y_val = load_images(df_val)
    X_test, y_test = load_images(df_test)

    print("X_train:", X_train.shape, X_train.dtype, "min/max:", X_train.min(), X_train.max())
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)

Guardamos los arreglos ya preprocesados en disco (fuera del repositorio, en `data/processed/`, ignorado por `.gitignore`) para no tener que releer y redimensionar las ~17,400 imágenes cada vez que se abra el notebook.

In [ ]:
if EJECUTAR_ORIGINAL:
    PROCESSED_DIR = Path("data/processed")
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    np.save(PROCESSED_DIR / "X_train.npy", X_train)
    np.save(PROCESSED_DIR / "y_train.npy", y_train)
    np.save(PROCESSED_DIR / "X_val.npy", X_val)
    np.save(PROCESSED_DIR / "y_val.npy", y_val)
    np.save(PROCESSED_DIR / "X_test.npy", X_test)
    np.save(PROCESSED_DIR / "y_test.npy", y_test)

    with open(PROCESSED_DIR / "label_mapping.json", "w") as f:
        json.dump(idx_to_label, f, indent=2)

    print("Arreglos preprocesados guardados en:", PROCESSED_DIR.resolve())

**Verificación visual del preprocesamiento**: comparamos una imagen original (200x200) contra su versión preprocesada (64x64, normalizada) para confirmar que la seña sigue siendo reconocible después de reducir la resolución.

In [ ]:
if EJECUTAR_ORIGINAL:
    example_row = df_train.iloc[0]
    original = Image.open(example_row["filepath"]).convert("RGB")
    processed = (X_train[0] * 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(original)
    axes[0].set_title(f"Original 200x200 — clase '{example_row['label']}'")
    axes[0].axis("off")
    axes[1].imshow(processed)
    axes[1].set_title(f"Preprocesada {IMG_SIZE}x{IMG_SIZE}, normalizada")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

**Hallazgo:** al reducir a 64x64 la forma general de la mano y la posición de los dedos siguen siendo claramente identificables, aunque se pierde algo de detalle fino, justo el tipo de detalle que distingue a letras como M/N/S o U/V/R (sección 2.3). Esto se retomará en la discusión de resultados de los modelos.

## Ejercicio 4: Modelos de Deep Learning (CNN)

Entrenamos **dos arquitecturas CNN distintas** sobre `X_train/y_train`, validando con `X_val/y_val`, para comparar su efectividad:

- **CNN-A (baseline):** dos bloques `Conv2D + MaxPooling`, seguidos de una capa densa y `Dropout`.
- **CNN-B (más profunda y regularizada):** tres bloques convolucionales con `BatchNormalization` y `Dropout`, para ver si mejora la generalización frente al baseline.

Usamos `EarlyStopping` monitoreando `val_accuracy` para no sobre-entrenar y controlar el tiempo de cómputo.

In [ ]:
if EJECUTAR_ORIGINAL:
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks

    NUM_CLASSES = len(CLASSES)
    EPOCHS = 15
    BATCH_SIZE = 128

    def make_early_stop():
        return callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)

    def build_cnn_a():
        model = keras.Sequential([
            layers.Input((IMG_SIZE, IMG_SIZE, 3)),
            layers.Conv2D(32, 3, activation="relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, activation="relu"),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.3),
            layers.Dense(NUM_CLASSES, activation="softmax"),
        ], name="CNN_A_baseline")
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    cnn_a = build_cnn_a()
    cnn_a.summary()

In [ ]:
if EJECUTAR_ORIGINAL:
    history_a = cnn_a.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[make_early_stop()], verbose=2,
    )

In [ ]:
if EJECUTAR_ORIGINAL:
    def build_cnn_b():
        # momentum=0.9 (en vez del 0.99 por defecto) para que las estadísticas de
        # BatchNormalization converjan más rápido dado el número de pasos por época;
        # con el valor por defecto la validación se queda "atascada" en accuracy
        # aleatoria durante varias épocas hasta que el promedio móvil se estabiliza.
        def bn():
            return layers.BatchNormalization(momentum=0.9)

        model = keras.Sequential([
            layers.Input((IMG_SIZE, IMG_SIZE, 3)),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            bn(),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(),
            layers.Dropout(0.25),

            layers.Conv2D(64, 3, padding="same", activation="relu"),
            bn(),
            layers.Conv2D(64, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(),
            layers.Dropout(0.25),

            layers.Conv2D(128, 3, padding="same", activation="relu"),
            bn(),
            layers.MaxPooling2D(),
            layers.Dropout(0.3),

            layers.Flatten(),
            layers.Dense(256, activation="relu"),
            bn(),
            layers.Dropout(0.4),
            layers.Dense(NUM_CLASSES, activation="softmax"),
        ], name="CNN_B_regularizada")
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    cnn_b = build_cnn_b()
    cnn_b.summary()

In [ ]:
if EJECUTAR_ORIGINAL:
    history_b = cnn_b.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[make_early_stop()], verbose=2,
    )

In [ ]:
if EJECUTAR_ORIGINAL:
    def plot_history(histories, labels, title=""):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for h, lbl in zip(histories, labels):
            axes[0].plot(h.history["accuracy"], label=f"{lbl} train")
            axes[0].plot(h.history["val_accuracy"], linestyle="--", label=f"{lbl} val")
            axes[1].plot(h.history["loss"], label=f"{lbl} train")
            axes[1].plot(h.history["val_loss"], linestyle="--", label=f"{lbl} val")
        axes[0].set_title(f"Accuracy {title}"); axes[0].set_xlabel("Época"); axes[0].legend(fontsize=8)
        axes[1].set_title(f"Loss {title}"); axes[1].set_xlabel("Época"); axes[1].legend(fontsize=8)
        plt.tight_layout()
        plt.show()

    plot_history([history_a, history_b], ["CNN-A", "CNN-B"], title="(CNN-A vs CNN-B)")

In [ ]:
if EJECUTAR_ORIGINAL:
    test_loss_a, test_acc_a = cnn_a.evaluate(X_test, y_test, verbose=0)
    test_loss_b, test_acc_b = cnn_b.evaluate(X_test, y_test, verbose=0)

    print(f"CNN-A -> test accuracy: {test_acc_a:.4f}, test loss: {test_loss_a:.4f}")
    print(f"CNN-B -> test accuracy: {test_acc_b:.4f}, test loss: {test_loss_b:.4f}")

    best_cnn_name = "CNN-A" if test_acc_a >= test_acc_b else "CNN-B"
    best_cnn_model = cnn_a if test_acc_a >= test_acc_b else cnn_b
    best_cnn_test_acc = max(test_acc_a, test_acc_b)
    print("\nMejor CNN:", best_cnn_name, f"(test accuracy = {best_cnn_test_acc:.4f})")

**Hallazgo:** ambas arquitecturas aprenden bien, pero **CNN-B (más profunda y regularizada) supera a CNN-A (baseline)**: 95.52% vs. 92.72% de accuracy en el set de prueba. Las curvas de entrenamiento muestran que CNN-B alcanza un accuracy de entrenamiento más alto gracias a `BatchNormalization` + `Dropout`, que compensan su mayor capacidad. `CNN-A` converge de forma más suave pero se estanca antes. A continuación revisamos la matriz de confusión de **CNN-B** (el mejor modelo de este ejercicio), prestando especial atención a los grupos de letras identificados como visualmente confundibles en el EDA (M/N/S y U/V/R).

In [ ]:
if EJECUTAR_ORIGINAL:
    from sklearn.metrics import confusion_matrix, classification_report

    y_pred_probs_cnn = best_cnn_model.predict(X_test, verbose=0)
    y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)

    cm = confusion_matrix(y_test, y_pred_cnn)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f"Matriz de confusión — {best_cnn_name} (set de prueba)")
    plt.xlabel("Predicción"); plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

    print(classification_report(y_test, y_pred_cnn, target_names=CLASSES, zero_division=0))

**Hallazgo:** el reporte de clasificación confirma, en buena medida, lo anticipado en el EDA (sección 2.3), aunque no exactamente con las mismas letras que esperábamos como las más débiles:

- El grupo **U/V/W** concentra los errores más notorios: **W tiene precision de solo 0.79** (muchas señas de otras letras del mismo estilo —dedos extendidos— se clasifican incorrectamente como W), mientras que U (0.94/0.98) y V (0.93/0.93) también muestran algo de confusión cruzada. Esto es consistente con el grupo U/V/R señalado en el EDA, aunque en la práctica el modelo confunde más con W que con R.
- El grupo **M/N** también muestra confusión moderada (M: 0.95 precisión/0.93 recall; N: 0.92/0.96), igual que anticipamos.
- Dos clases que **no** habíamos identificado en el EDA resultaron ser las más difíciles de todas: **`nothing`** (recall de solo 0.67 — el modelo frecuentemente no reconoce una "no-seña" como tal) y **Z** (precision 0.85, aunque recall 1.00). También S (0.89/0.94) y E (0.98/0.88) tienen cierta dificultad.
- El resto de las clases tiene f1-score ≥ 0.94, incluyendo letras "fáciles" como B, C, D, F, P, Q.

Esto confirma parcialmente la hipótesis del EDA (los grupos M/N y U/V sí generan confusión), pero también muestra que **la resolución reducida (64x64) puede generar dificultades adicionales no anticipadas** en clases que a simple vista parecían fáciles de distinguir (como `nothing` o Z), reforzando la importancia de revisar la matriz de confusión real en vez de basarse solo en la intuición del EDA.

## Ejercicio 5: Red neuronal simple (fully-connected)

Probamos **dos variantes** de una red totalmente conectada (sin capas convolucionales, aplanando la imagen directamente) para compararlas contra las CNNs del Ejercicio 4:

- **FC-A (pequeña):** dos capas densas (128, 64).
- **FC-B (grande):** tres capas densas (512, 256, 128).

In [ ]:
if EJECUTAR_ORIGINAL:
    def build_fc(hidden_layers, dropout=0.3, name="FC"):
        # BatchNormalization justo después de Flatten: sin ella, la red fully-connected
        # no lograba entrenar (el optimizador la llevaba a una salida uniforme -log(1/29)-
        # desde la primera época, un caso de "dead ReLU" típico con entradas de muy alta
        # dimensión sin normalizar por lote).
        model = keras.Sequential([
            layers.Input((IMG_SIZE, IMG_SIZE, 3)),
            layers.Flatten(),
            layers.BatchNormalization(),
        ], name=name)
        for units in hidden_layers:
            model.add(layers.Dense(units, activation="relu"))
            model.add(layers.Dropout(dropout))
        model.add(layers.Dense(NUM_CLASSES, activation="softmax"))
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    fc_a = build_fc([128, 64], dropout=0.3, name="FC_A_pequena")
    fc_a.summary()
    history_fc_a = fc_a.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[make_early_stop()], verbose=2,
    )

In [ ]:
if EJECUTAR_ORIGINAL:
    fc_b = build_fc([512, 256, 128], dropout=0.4, name="FC_B_grande")
    fc_b.summary()
    history_fc_b = fc_b.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[make_early_stop()], verbose=2,
    )

In [ ]:
if EJECUTAR_ORIGINAL:
    plot_history([history_fc_a, history_fc_b], ["FC-A", "FC-B"], title="(FC-A vs FC-B)")

    test_loss_fc_a, test_acc_fc_a = fc_a.evaluate(X_test, y_test, verbose=0)
    test_loss_fc_b, test_acc_fc_b = fc_b.evaluate(X_test, y_test, verbose=0)

    print(f"FC-A -> test accuracy: {test_acc_fc_a:.4f}")
    print(f"FC-B -> test accuracy: {test_acc_fc_b:.4f}")

    best_fc_name = "FC-A" if test_acc_fc_a >= test_acc_fc_b else "FC-B"
    best_fc_model = fc_a if test_acc_fc_a >= test_acc_fc_b else fc_b
    best_fc_test_acc = max(test_acc_fc_a, test_acc_fc_b)
    print("\nMejor red fully-connected:", best_fc_name, f"(test accuracy = {best_fc_test_acc:.4f})")

**Hallazgo:** ambas redes fully-connected quedan muy por debajo de las CNN del Ejercicio 4: **FC-A alcanza 53.1%** y **FC-B solo 42.0%** de accuracy en el set de prueba, frente al 92.7%–95.5% de las CNN. Dos observaciones interesantes:

1. **Las CNN superan ampliamente a las redes FC**, confirmando la hipótesis: las capas convolucionales explotan la estructura espacial de la imagen (comparten pesos entre regiones, son invariantes a pequeñas traslaciones), mientras que una red FC trata cada uno de los 12,288 píxeles como una variable independiente y necesita aprender desde cero cualquier noción de "forma" o "vecindad" entre píxeles.
2. **FC-B (la red más grande, con 3 capas de 512-256-128) generaliza peor que FC-A (la más pequeña, 128-64)**, a pesar de tener más parámetros y capacidad. Esto sugiere que, con solo 12,180 imágenes de entrenamiento, una red FC grande sin la restricción estructural de las convoluciones tiene más facilidad para sobre-ajustarse a ruido de píxeles que para aprender un patrón que generalice, incluso con `Dropout` y `BatchNormalization` de por medio.

## Ejercicio 6: Otro algoritmo — Random Forest

**Justificación de la selección:** elegimos **Random Forest** porque (a) no requiere GPU y entrena rápido en CPU, (b) es robusto a ruido/outliers y no necesita mucho tuneo para dar un resultado razonable, y (c) nos sirve como referencia de un modelo de Machine Learning "clásico" para dimensionar cuánto aportan realmente las redes profundas frente a un enfoque más simple.

Los píxeles aplanados son un espacio de muy alta dimensión (64×64×3 = 12,288 variables) y muy correlacionado entre sí (píxeles vecinos), así que antes de entrenar el bosque reducimos la dimensionalidad con **PCA**, ajustado únicamente sobre el conjunto de entrenamiento.

In [ ]:
if EJECUTAR_ORIGINAL:
    from sklearn.decomposition import PCA
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    X_train_flat = X_train.reshape(len(X_train), -1)
    X_val_flat = X_val.reshape(len(X_val), -1)
    X_test_flat = X_test.reshape(len(X_test), -1)

    N_COMPONENTS = 150
    pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train_flat)
    X_val_pca = pca.transform(X_val_flat)
    X_test_pca = pca.transform(X_test_flat)

    print(f"Varianza explicada acumulada con {N_COMPONENTS} componentes: {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
if EJECUTAR_ORIGINAL:
    rf_configs = {
        "RF-A (100 árboles, sin límite de profundidad)": RandomForestClassifier(
            n_estimators=100, max_depth=None, random_state=SEED, n_jobs=-1
        ),
        "RF-B (300 árboles, profundidad máx. 30)": RandomForestClassifier(
            n_estimators=300, max_depth=30, random_state=SEED, n_jobs=-1
        ),
    }

    rf_results = {}
    for name, clf in rf_configs.items():
        clf.fit(X_train_pca, y_train)
        val_acc = accuracy_score(y_val, clf.predict(X_val_pca))
        rf_results[name] = {"model": clf, "val_accuracy": val_acc}
        print(f"{name} -> val accuracy: {val_acc:.4f}")

    best_rf_name = max(rf_results, key=lambda k: rf_results[k]["val_accuracy"])
    best_rf_model = rf_results[best_rf_name]["model"]
    test_acc_rf = accuracy_score(y_test, best_rf_model.predict(X_test_pca))

    print(f"\nMejor Random Forest: {best_rf_name}")
    print(f"Test accuracy: {test_acc_rf:.4f}")

**Hallazgo:** el mejor Random Forest (RF-B, 300 árboles, profundidad máxima 30) obtiene **89.73%** de accuracy en el set de prueba, un resultado intermedio: muy por encima de las redes fully-connected (42.0%–53.1%) y razonablemente cerca de CNN-A (92.7%), aunque por debajo de CNN-B (95.5%). Esto tiene sentido: PCA + Random Forest sí logra capturar patrones globales útiles de la imagen (con 150 componentes ya se conserva el 96.2% de la varianza), pero al aplanar y proyectar la imagen se pierden las relaciones espaciales locales (qué píxeles son vecinos de cuáles) que las convoluciones sí aprovechan directamente. Aun así, es una alternativa muy razonable si no se dispone de GPU: se entrena en segundos/minutos (frente a los ~10-15 minutos de las CNN en CPU) y con muy poco ajuste de hiperparámetros ya da un resultado competitivo, superando incluso a ambas redes fully-connected.

## Comparación general de modelos (sin augmentation)

In [ ]:
if EJECUTAR_ORIGINAL:
    comparison = pd.DataFrame({
        "modelo": ["CNN-A", "CNN-B", "FC-A", "FC-B", best_rf_name],
        "tipo": ["CNN", "CNN", "Fully-connected", "Fully-connected", "Random Forest"],
        "test_accuracy": [test_acc_a, test_acc_b, test_acc_fc_a, test_acc_fc_b, test_acc_rf],
    }).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(9, 5))
    sns.barplot(data=comparison, x="modelo", y="test_accuracy", hue="tipo", dodge=False)
    plt.ylim(0, 1)
    plt.title("Comparación de accuracy en el set de prueba")
    plt.ylabel("Test accuracy")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    comparison

## Ejercicio 7: Image augmentation

Aplicamos data augmentation a la arquitectura CNN que resultó ganadora en el Ejercicio 4 (`best_cnn_name`) y la re-entrenamos desde cero para ver si mejora su capacidad de generalización.

**Transformaciones utilizadas:** rotación leve (± ~15°), zoom, traslación pequeña y variación de contraste. Estas transformaciones son razonables para señas porque simulan variaciones realistas de cómo una persona sostiene la mano frente a la cámara (ángulo, distancia, iluminación) sin cambiar la identidad de la seña.

**Transformación deliberadamente excluida: `flip` horizontal.** No lo usamos porque:

- **¿Por qué un flip horizontal podría cambiar el significado de una seña en vez de solo aumentar los datos?** Muchas letras del alfabeto ASL se distinguen precisamente por la posición relativa de los dedos y el pulgar (por ejemplo, M/N/S dependen de bajo cuántos dedos queda el pulgar). Al voltear horizontalmente la imagen, esa relación espacial se invierte como en un espejo: una mano que hace una "seña válida" para una letra puede terminar pareciendo la configuración de otra letra distinta, o una configuración que ninguna persona real haría con esa mano. En otras palabras, el flip no genera una nueva muestra *de la misma clase*, sino potencialmente una muestra de *otra clase* o una muestra inválida, lo cual introduce etiquetas incorrectas en el entrenamiento en vez de aumentar la variabilidad legítima de la clase original.
- **¿Qué transformaciones sí tienen sentido para señas y cuáles no?** Tienen sentido las transformaciones que imitan variaciones reales de cómo se captura la seña sin alterar la forma de la mano: rotaciones pequeñas, zoom/traslación (distancia y encuadre respecto a la cámara), y cambios de brillo/contraste (iluminación). No tienen sentido las transformaciones que alteran la geometría de la mano en sí o que no ocurrirían en la práctica: el flip horizontal (cambia qué dedos están de qué lado), el flip vertical (dejaría la mano "de cabeza", algo que no ocurre en un uso normal frente a una cámara), y rotaciones grandes (>45°), que no reflejan cómo alguien sostendría la mano frente a una cámara en una aplicación de escritura en tiempo real.

In [ ]:
if EJECUTAR_ORIGINAL:
    augmentation = keras.Sequential([
        layers.RandomRotation(0.04),        # ~ ±15 grados
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.1),
    ], name="augmentation")

    build_best_cnn = build_cnn_a if best_cnn_name == "CNN-A" else build_cnn_b

    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    outputs = build_best_cnn()(x)
    cnn_aug = keras.Model(inputs, outputs, name=f"{best_cnn_name.replace('-', '_')}_con_augmentation")
    cnn_aug.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history_aug = cnn_aug.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[make_early_stop()], verbose=2,
    )

In [ ]:
if EJECUTAR_ORIGINAL:
    plot_history([history_aug], [f"{best_cnn_name}+aug"], title="(con augmentation)")

    test_loss_aug, test_acc_aug = cnn_aug.evaluate(X_test, y_test, verbose=0)
    print(f"{best_cnn_name} sin augmentation -> test accuracy: {best_cnn_test_acc:.4f}")
    print(f"{best_cnn_name} con augmentation  -> test accuracy: {test_acc_aug:.4f}")

**Hallazgo:** en esta corrida, CNN-B **con augmentation** (95.59%) y **sin augmentation** (95.52%) terminaron prácticamente empatadas — una diferencia de menos de un punto décimo porcentual, dentro del margen de variación normal que se observa entre corridas de entrenamiento en CPU (sin fijar semilla de TensorFlow, hay algo de aleatoriedad en el orden de batches y en operaciones no determinísticas). Es decir, en este dataset la augmentation **ni ayudó ni perjudicó de forma significativa** al accuracy interno. Esto es coherente con algo que ya habíamos anticipado en el EDA (Ejercicio 1): las imágenes de este dataset provienen de sesiones de captura muy controladas y homogéneas (mismo fondo/iluminación/encuadre dentro de cada clase), así que nuestro propio set de prueba —al ser un subconjunto aleatorio de esas mismas sesiones— es igual de homogéneo, y por lo tanto no había mucho "gap" de generalización que la augmentation pudiera cerrar. El beneficio esperado de la augmentation (mejor generalización a condiciones nunca vistas, como fondos/ángulos distintos) es precisamente lo que evaluamos con imágenes independientes en el Ejercicio 8.

## Selección del mejor modelo

Comparamos **todos** los modelos entrenados (con y sin augmentation) y seleccionamos el de mayor accuracy en el set de prueba como el modelo final para el Ejercicio 8.

In [ ]:
if EJECUTAR_ORIGINAL:
    final_comparison = pd.concat([
        comparison,
        pd.DataFrame([{
            "modelo": f"{best_cnn_name}+aug",
            "tipo": "CNN + augmentation",
            "test_accuracy": test_acc_aug,
        }])
    ], ignore_index=True).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

    display(final_comparison)

    model_registry = {
        "CNN-A": cnn_a,
        "CNN-B": cnn_b,
        "FC-A": fc_a,
        "FC-B": fc_b,
        best_rf_name: best_rf_model,
        f"{best_cnn_name}+aug": cnn_aug,
    }

    final_model_name = final_comparison.iloc[0]["modelo"]
    final_model = model_registry[final_model_name]
    final_test_acc = final_comparison.iloc[0]["test_accuracy"]

    print(f"\nModelo final seleccionado: {final_model_name} (test accuracy = {final_test_acc:.4f})")

Guardamos los modelos entrenados en disco (`data/models/`, fuera del repositorio) para no tener que volver a entrenarlos cada vez que se agreguen o cambien fotos en `imagenes_grupo/` — solo haría falta re-ejecutar desde el Ejercicio 8 en adelante.

In [ ]:
if EJECUTAR_ORIGINAL:
    import joblib

    MODELS_DIR = Path("data/models")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    for name, obj in {"cnn_a": cnn_a, "cnn_b": cnn_b, "fc_a": fc_a, "fc_b": fc_b, "cnn_aug": cnn_aug}.items():
        obj.save(MODELS_DIR / f"{name}.keras")

    joblib.dump(best_rf_model, MODELS_DIR / "best_rf_model.joblib")
    joblib.dump(pca, MODELS_DIR / "pca.joblib")
    joblib.dump({"best_cnn_name": best_cnn_name, "best_rf_name": best_rf_name,
                 "final_model_name": final_model_name}, MODELS_DIR / "metadata.joblib")

    print("Modelos guardados en:", MODELS_DIR.resolve())

## Ejercicio 8: Pruebas con señas hechas a mano por el equipo

Probamos el mejor modelo con **fotos reales** hechas por los 3 integrantes del grupo (5 letras distintas por integrante, 15 fotos en total), en la carpeta `imagenes_grupo/`:

- **luis:** A, B, C, D, G
- **bran:** K, L, M, N, R
- **david:** S, U, V, W, space

Cada archivo sigue la convención de nombres **`<LETRA>_<identificador>.jpg`** (por ejemplo `A_luis.jpeg`), donde la letra antes del primer guión bajo es la etiqueta real (ground truth) usada al evaluar. Para agregar más fotos o reemplazar alguna, basta con seguir la misma convención y volver a correr desde esta sección — los modelos ya están guardados en `data/models/`, así que **no hace falta reentrenar nada** (ver la celda de recarga justo debajo).

**Requisitos que se usaron para tomar las fotos** (referencia para futuras fotos):
- Formato `.jpg`/`.png`, una sola mano centrada ocupando la mayor parte del encuadre, fondo simple, buena luz frontal, resolución mínima ~200x200 píxeles, en color (RGB).

**Recarga de modelos sin reentrenar:** esta celda carga los modelos y el set de prueba ya guardados en `data/models/` y `data/processed/` (generados la primera vez que se corrió el notebook completo). Así, para probar fotos nuevas en `imagenes_grupo/` basta con ejecutar el notebook **a partir de aquí**, sin repetir el entrenamiento de los Ejercicios 4-7.

In [ ]:
if EJECUTAR_ORIGINAL:
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from PIL import Image
    import matplotlib.pyplot as plt
    import json
    import joblib
    from tensorflow import keras
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    MODELS_DIR = Path("data/models")
    PROCESSED_DIR = Path("data/processed")

    IMG_SIZE = 64
    idx_to_label = {int(k): v for k, v in json.load(open(PROCESSED_DIR / "label_mapping.json")).items()}
    CLASSES = [idx_to_label[i] for i in range(len(idx_to_label))]

    X_test = np.load(PROCESSED_DIR / "X_test.npy")
    y_test = np.load(PROCESSED_DIR / "y_test.npy")

    cnn_a = keras.models.load_model(MODELS_DIR / "cnn_a.keras")
    cnn_b = keras.models.load_model(MODELS_DIR / "cnn_b.keras")
    fc_a = keras.models.load_model(MODELS_DIR / "fc_a.keras")
    fc_b = keras.models.load_model(MODELS_DIR / "fc_b.keras")
    cnn_aug = keras.models.load_model(MODELS_DIR / "cnn_aug.keras")
    best_rf_model = joblib.load(MODELS_DIR / "best_rf_model.joblib")
    pca = joblib.load(MODELS_DIR / "pca.joblib")

    metadata = joblib.load(MODELS_DIR / "metadata.joblib")
    best_cnn_name = metadata["best_cnn_name"]
    best_rf_name = metadata["best_rf_name"]

    # Recalculamos el accuracy de cada modelo sobre el mismo test set guardado
    # (solo inferencia, toma segundos) para reconstruir la tabla comparativa.
    X_test_pca = pca.transform(X_test.reshape(len(X_test), -1))

    _, test_acc_a = cnn_a.evaluate(X_test, y_test, verbose=0)
    _, test_acc_b = cnn_b.evaluate(X_test, y_test, verbose=0)
    _, test_acc_fc_a = fc_a.evaluate(X_test, y_test, verbose=0)
    _, test_acc_fc_b = fc_b.evaluate(X_test, y_test, verbose=0)
    _, test_acc_aug = cnn_aug.evaluate(X_test, y_test, verbose=0)
    test_acc_rf = accuracy_score(y_test, best_rf_model.predict(X_test_pca))

    comparison = pd.DataFrame({
        "modelo": ["CNN-A", "CNN-B", "FC-A", "FC-B", best_rf_name],
        "tipo": ["CNN", "CNN", "Fully-connected", "Fully-connected", "Random Forest"],
        "test_accuracy": [test_acc_a, test_acc_b, test_acc_fc_a, test_acc_fc_b, test_acc_rf],
    }).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

    final_comparison = pd.concat([
        comparison,
        pd.DataFrame([{"modelo": f"{best_cnn_name}+aug", "tipo": "CNN + augmentation", "test_accuracy": test_acc_aug}]),
    ], ignore_index=True).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

    model_registry = {
        "CNN-A": cnn_a, "CNN-B": cnn_b, "FC-A": fc_a, "FC-B": fc_b,
        best_rf_name: best_rf_model, f"{best_cnn_name}+aug": cnn_aug,
    }
    final_model_name = final_comparison.iloc[0]["modelo"]
    final_model = model_registry[final_model_name]

    print("Modelos y datos recargados desde disco.")
    print(f"Modelo final: {final_model_name} (test accuracy = {final_comparison.iloc[0]['test_accuracy']:.4f})")

In [ ]:
if EJECUTAR_ORIGINAL:
    IMAGENES_GRUPO_DIR = Path("imagenes_grupo")

    def preprocess_single_image(filepath, img_size=IMG_SIZE):
        img = Image.open(filepath).convert("RGB").resize((img_size, img_size))
        return np.asarray(img, dtype=np.float32) / 255.0

    def predict_with_final_model(arr):
        if isinstance(final_model, RandomForestClassifier):
            flat_pca = pca.transform(arr.reshape(1, -1))
            pred_idx = final_model.predict(flat_pca)[0]
        else:
            pred_idx = int(np.argmax(final_model.predict(arr[np.newaxis, ...], verbose=0)[0]))
        return idx_to_label[pred_idx]

    own_files = sorted(IMAGENES_GRUPO_DIR.glob("*.*"))
    assert len(own_files) >= 5, "Se requieren al menos 5 imágenes en imagenes_grupo/"
    print(f"Imágenes encontradas en {IMAGENES_GRUPO_DIR}/: {len(own_files)}")

    own_results = []
    for f in own_files:
        true_label = f.stem.split("_")[0]
        arr = preprocess_single_image(f)
        pred_label = predict_with_final_model(arr)
        own_results.append({
            "archivo": f.name, "real": true_label, "prediccion": pred_label,
            "correcto": true_label.upper() == pred_label.upper(),
        })

    df_own = pd.DataFrame(own_results)
    df_own

In [ ]:
if EJECUTAR_ORIGINAL:
    n = len(own_files)
    cols = min(n, 5)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, f, row in zip(axes, own_files, df_own.itertuples()):
        img = Image.open(f).convert("RGB")
        ax.imshow(img)
        color = "green" if row.correcto else "red"
        ax.set_title(f"real: {row.real} | pred: {row.prediccion}", color=color, fontsize=10)
        ax.axis("off")
    for ax in axes[len(own_files):]:
        ax.axis("off")

    plt.suptitle(f"Predicciones del modelo final ({final_model_name}) sobre imagenes_grupo/", y=1.02)
    plt.tight_layout()
    plt.show()

    accuracy_own = df_own["correcto"].mean()
    print(f"Accuracy sobre imágenes propias del equipo: {accuracy_own:.2%} ({df_own['correcto'].sum()}/{len(df_own)})")

In [ ]:
if EJECUTAR_ORIGINAL:
    # Chequeo adicional: ¿el modelo con mejor accuracy interno (test set propio) es
    # también el que mejor generaliza a estas imágenes independientes? Evaluamos TODOS
    # los modelos entrenados sobre imagenes_grupo/, no solo el seleccionado como "final".

    def predict_label_generic(model_obj, arr):
        if isinstance(model_obj, RandomForestClassifier):
            pred_idx = model_obj.predict(pca.transform(arr.reshape(1, -1)))[0]
        else:
            pred_idx = int(np.argmax(model_obj.predict(arr[np.newaxis, ...], verbose=0)[0]))
        return idx_to_label[pred_idx]

    own_arrays = [preprocess_single_image(f) for f in own_files]
    own_true = [f.stem.split("_")[0] for f in own_files]

    generalization_rows = []
    for name, model_obj in model_registry.items():
        preds = [predict_label_generic(model_obj, arr) for arr in own_arrays]
        acc_placeholder = np.mean([p.upper() == t.upper() for p, t in zip(preds, own_true)])
        generalization_rows.append({"modelo": name, "test_accuracy_interno": final_comparison.set_index("modelo").loc[name, "test_accuracy"] if name in final_comparison["modelo"].values else np.nan,
                                     "accuracy_imagenes_grupo": acc_placeholder})

    df_generalization = pd.DataFrame(generalization_rows).sort_values("accuracy_imagenes_grupo", ascending=False).reset_index(drop=True)
    df_generalization

**Hallazgo clave:** con fotos **reales**, el panorama cambia por completo frente a los placeholders. La clasificación por `accuracy_imagenes_grupo` ahora es muy baja para **todos** los modelos: CNN-B+aug (el modelo final) apenas 20% (3/15), CNN-A y CNN-B 13.3% cada una, FC-A y FC-B 6.7%, y Random Forest 0%. Es decir, el **orden relativo entre modelos se mantiene** (las CNN siguen siendo las mejores, Random Forest el peor), pero la magnitud del accuracy colapsa para todos por igual. Esto es una señal fuerte de que el problema **no es que un modelo sea mejor que otro**, sino que **las 15 fotos reales vienen de una distribución muy distinta a la de entrenamiento** — precisamente el riesgo de generalización que no se alcanzaba a ver con los placeholders (que, al ser parte del mismo dataset, no exponían este problema). Investigamos la causa concreta en el hallazgo de la siguiente celda.

**Hallazgo (con fotos reales del equipo):** el modelo final (`CNN-B+aug`) solo clasificó correctamente **3 de 15** fotos reales (20%): acertó C (luis), K y L (bran); falló en las 12 restantes, incluyendo confusiones poco intuitivas (S, U, V y W predichas como "E"). Al revisar las fotos directamente encontramos una causa muy concreta, más allá de solo "condiciones distintas":

- **Relación de aspecto:** las fotos de entrenamiento son cuadradas (200×200, relación 1:1), pero las fotos reales del equipo son verticales tipo retrato — las de luis/bran miden 1200×1600 (relación 3:4) y las de david 720×1280 (relación 9:16). Nuestro preprocesamiento (`resize((64, 64))`) **no recorta, solo redimensiona**, así que una foto 3:4 o 9:16 se comprime horizontalmente para encajar en un cuadrado, distorsionando la forma de la mano; cuanto más alargada la foto (caso de david, 9:16), más severa la distorsión.
- **Proporción mano/fondo:** en el dataset original la mano ocupa casi todo el cuadro; en las fotos reales (tomadas con el brazo extendido frente a una pared o una cama) la mano es una fracción pequeña de una foto vertical mucho más grande, rodeada de fondo (pared, sábana). Al redimensionar a 64×64, el detalle real de los dedos se reduce muchísimo más que en las imágenes de entrenamiento.
- **Fondo e iluminación distintos**, como ya anticipábamos en el EDA y en el Ejercicio 9: paredes texturizadas o una sábana, en vez del fondo uniforme del set de captura original.

## Ejercicio 9: Reflexión de accesibilidad y sesgo

**¿Qué limitaciones tiene este dataset para un producto real de accesibilidad como el de SignBridge?**

- **Tono de piel:** en el EDA (Ejercicio 1, sección 2.1) observamos que las imágenes de una misma clase comparten fondo e iluminación consistentes entre sí, lo cual sugiere que fueron capturadas en pocas sesiones con probablemente muy pocas personas distintas. Esto implica poca diversidad de tonos de piel, y un modelo entrenado así puede funcionar peor para usuarios con tonos de piel no representados en el dataset (un sesgo de representación clásico en visión por computadora).
- **Ángulo de cámara:** todas las fotos parecen tomadas desde un ángulo frontal fijo y a una distancia similar. En un producto real, los usuarios sostendrían el celular en ángulos y distancias muy variables, y el modelo podría fallar ante perspectivas no vistas en entrenamiento.
- **Iluminación:** el fondo e iluminación son homogéneos dentro del dataset (probablemente un entorno controlado tipo estudio/laboratorio). Un entorno real (casa, calle, luz natural variable, contraluz) introduce condiciones muy distintas a las vistas en entrenamiento.
- **Tamaño y forma de la mano:** al provenir de pocas personas, el dataset probablemente no cubre bien la variabilidad de tamaños de mano (niños vs. adultos), ni condiciones como manos con alguna diferencia física, anillos, uñas largas, etc., que sí aparecerían en una población real de usuarios.
- **Señas estáticas únicamente:** el dataset representa J y Z (que en ASL real son señas con movimiento) como poses estáticas, lo cual no captura el componente dinámico real de esas letras.

**¿Qué haría falta para llevar este prototipo a un caso de uso real como el de SignBridge?**

- Recolectar (o complementar con) datos de **muchas más personas**, con distintos tonos de piel, edades, tamaños de mano y, si es posible, con alguna diversidad de habilidad motriz.
- Capturar imágenes/video en **condiciones variadas** de fondo, iluminación y ángulo de cámara (idealmente con las cámaras de celular reales que usaría la app, no cámaras de estudio), incluyendo el tipo de fotos que tomamos en el Ejercicio 8.
- Pasar de imágenes estáticas a **video/secuencias** para soportar letras dinámicas (J, Z) y, eventualmente, palabras completas en vez de solo letras sueltas.
- Evaluar el modelo de forma desagregada (por tono de piel, por condición de iluminación, etc.) y no solo con un accuracy global, para detectar sesgos de desempeño antes de lanzar el producto — como recomendación concreta, mediríamos el accuracy del modelo final por subgrupos (ej. distintos fondos/iluminación en las fotos del Ejercicio 8, y a futuro por tono de piel) para verificar que ningún grupo tenga un desempeño notablemente peor.
- Considerar el rendimiento en tiempo real en dispositivos móviles (latencia, tamaño del modelo) ya que el objetivo final es una app de traducción en tiempo real.

- **Recorte/detección de la mano antes de clasificar:** el Ejercicio 8 nos dio evidencia empírica directa de esto — con fotos reales del equipo (verticales, mano ocupando solo una parte del encuadre) el accuracy del mejor modelo cayó de 93.3% (14/15, con placeholders del propio dataset) a apenas 20% (3/15), en buena parte porque nuestro preprocesamiento asume implícitamente una foto ya cuadrada y recortada alrededor de la mano, como las del dataset de entrenamiento. Un producto real necesitaría un paso de detección/recorte de mano (por ejemplo con un modelo de detección de manos tipo MediaPipe, o guiando al usuario a encuadrar su mano dentro de un recuadro en la app) antes de pasar la imagen al clasificador.

## B. Implementación CUDA y admisión del entorno

Los kernels K01–K19 y las revisiones de rendimiento se documentan con su código.
Se mantienen FP32, CNN-B base, pesos iniciales, 17,400 imágenes y splits congelados,
batch128 y15épocas (1440actualizaciones). No se vuelve a descargar ni muestrear.

Los productos grandes usan cuBLAS SGEMM; sus grids internos los administra NVIDIA.
Los grids de im2col/col2im, pooling, BN, Dropout, softmax y Adam son explícitos.
La preparación CPU usa6workers y las transferencias se contabilizan.


In [ ]:
# Hardware REAL registrado durante la corrida, no una consulta nueva.
hardware_registrado = json.loads((ULTIMO_RUN / "hardware.json").read_text(encoding="utf8"))
print(json.dumps(hardware_registrado, indent=2, ensure_ascii=False))


## C. Corridas reales, sin sobrescribir evidencias

La ejecución automática establece `ASL_CUDA_RUN=1`. Cada corrida es un proceso
nuevo en PCUDA. Si la serie ya terminó, se conservan sus observaciones y se
reutilizan para mostrar resultados; no se repiten entrenamientos por abrir el
notebook. Una serie incompleta exige diagnóstico explícito.


In [ ]:
from benchmark_cuda import prepare, run_series, locations
config_path, results_path = locations(EXPERIMENTO)
if EJECUTAR_GPU:
    if not config_path.exists():
        prepare(EXPERIMENTO)
    serie = run_series(EXPERIMENTO)
    print(json.dumps(serie, indent=2, ensure_ascii=False))
else:
    print("Mediciones desactivadas. Configure ASL_CUDA_RUN=1 para crear o consultar la serie.")


## D. Resultados, transferencias y comparación

Se muestran datos reales de los RUN. La validación CPU está anidada en entrenamiento; no se suma dos veces. Los eventos no sustituyen tiempo de pared sincronizado.

In [ ]:
from IPython.display import display, Markdown, Image
if (results_path / "serie.json").is_file():
    from resumir_cuda import generar
    generar(EXPERIMENTO)
    display(Markdown((results_path / "resultados.md").read_text(encoding="utf8")))
    for figura in sorted((results_path / "graficas").glob("*.png")):
        display(Image(filename=str(figura)))
else:
    print("Todavía no existe una serie. No se inventan resultados.")


## E. Fuentes de la implementación para revisión

Los siguientes bloques presentan el código que importa el notebook. La fuente
ejecutable única permanece en los módulos; no se mantienen dos implementaciones
independientes. Sus hashes forman parte de la configuración congelada.


### __init__.py

```python
"""Explicit CUDA Python implementation of the original ASL CNN-B."""

```


### blas.py

```python
"""FP32 cuBLAS SGEMM on Numba-owned device memory and the SAME CUDA stream.

The installed NVIDIA CUDA toolkit supplies the DLL. No tensor cores, no TF32,
no NumPy/CPU multiplication. Geometry inside cuBLAS belongs to NVIDIA's backend.
"""
import ctypes as ct
import os
from pathlib import Path


class Blas:
    def __init__(self, stream):
        root = Path(os.environ["CUDA_HOME"])
        self.path = root / "bin/x64/cublas64_13.dll"
        self.dll = ct.WinDLL(str(self.path))
        self.handle = ct.c_void_p()
        self._signature("cublasCreate_v2", [ct.POINTER(ct.c_void_p)])
        self._signature("cublasSetStream_v2", [ct.c_void_p, ct.c_void_p])
        self._signature("cublasSetMathMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasSetAtomicsMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasGetVersion_v2", [ct.c_void_p, ct.POINTER(ct.c_int)])
        self._signature("cublasDestroy_v2", [ct.c_void_p])
        self._signature("cublasSgemm_v2", [ct.c_void_p, ct.c_int, ct.c_int,
            ct.c_int, ct.c_int, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int,
            ct.c_void_p, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int])
        self._call("cublasCreate_v2", ct.byref(self.handle))
        raw = stream.handle
        self._call("cublasSetStream_v2", self.handle, ct.c_void_p(raw.value if hasattr(raw, "value") else int(raw)))
        self._call("cublasSetMathMode", self.handle, 2)  # CUBLAS_PEDANTIC_MATH
        self._call("cublasSetAtomicsMode", self.handle, 0)  # CUBLAS_ATOMICS_NOT_ALLOWED
        version = ct.c_int()
        self._call("cublasGetVersion_v2", self.handle, ct.byref(version))
        self.version = version.value

    def _signature(self, name, args):
        fn = getattr(self.dll, name)
        fn.argtypes, fn.restype = args, ct.c_int

    def _call(self, name, *args):
        status = getattr(self.dll, name)(*args)
        if status:
            raise RuntimeError(f"{name}: cuBLAS status {status}")

    @staticmethod
    def ptr(array):
        return ct.c_void_p(array.device_ctypes_pointer.value)

    def gemm(self, a, b, out, M, N, K, transpose_a=False, transpose_b=False):
        """C_row=op(A)_row @ op(B)_row, implemented as C_col=B_col @ A_col."""
        alpha, beta = ct.c_float(1.), ct.c_float(0.)
        self._call("cublasSgemm_v2", self.handle, int(transpose_b), int(transpose_a),
                   N, M, K, ct.byref(alpha), self.ptr(b), K if transpose_b else N,
                   self.ptr(a), M if transpose_a else K, ct.byref(beta), self.ptr(out), N)

    def close(self):
        if self.handle:
            self._call("cublasDestroy_v2", self.handle)
            self.handle = ct.c_void_p()

```


### kernels.py

```python
"""K01-K19, DIS-CUDA-001: FP32 NHWC, gather backward, explicit reductions.

Flat buffers have explicit active dimensions; capacity never defines batch size.
No floating point atomics or cross-block barriers. Partial gradients are unique.
"""
import math
from numba import float32
from .runtime import cuda


@cuda.jit(device=True, inline=True)
def _left(a, mode, m, k, H, W, Ci, Co, I, O):
    if mode == 0:  # virtual im2col
        n = m // (H * W)
        h = (m // W) % H + k // (3 * Ci) - 1
        w = m % W + (k // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + k % Ci]
        return float32(0)
    if mode == 1:  # gather dX, spatially reversed offsets
        n = m // (H * W)
        h = (m // W) % H - k // (3 * Co) + 1
        w = m % W - (k // Co) % 3 + 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Co + k % Co]
        return float32(0)
    if mode == 2:  # im2col transpose without materializing it
        n = k // (H * W)
        h = (k // W) % H + m // (3 * Ci) - 1
        w = k % W + (m // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + m % Ci]
        return float32(0)
    if mode == 3:
        return a[m * I + k]
    if mode == 4:
        return a[m * O + k]
    return a[k * I + m]


@cuda.jit(device=True, inline=True)
def _right(v, mode, k, n, Ci, Co, I, O):
    if mode == 0 or mode == 2:
        return v[k * Co + n]
    if mode == 1:
        return v[((k // Co) * Ci + n) * Co + k % Co]
    if mode == 3 or mode == 5:
        return v[k * O + n]
    return v[n * O + k]


def _make_gemm(mode):
    @cuda.jit(cache=True, fastmath=False)
    def kernel(a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0):
        sa = cuda.shared.array((16, 16), float32)
        sb = cuda.shared.array((16, 16), float32)
        tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
        m, n = m0 + cuda.blockIdx.y * 16 + ty, cuda.blockIdx.x * 16 + tx
        begin, end = 0, K
        if mode == 2 or mode == 5:
            begin = cuda.blockIdx.z * 2048
            end = min(K, begin + 2048)
        acc = float32(0)
        for k0 in range(begin, end, 16):
            sa[ty, tx] = float32(0)
            sb[ty, tx] = float32(0)
            if m < M and k0 + tx < end:
                sa[ty, tx] = _left(a, mode, m, k0 + tx, H, W, Ci, Co, I, O)
            if n < N and k0 + ty < end:
                sb[ty, tx] = _right(v, mode, k0 + ty, n, Ci, Co, I, O)
            cuda.syncthreads()
            for kk in range(16):
                acc += sa[ty, kk] * sb[kk, tx]
            cuda.syncthreads()
        if m < M and n < N:
            index = m * N + n
            if mode == 2 or mode == 5:
                index += cuda.blockIdx.z * M * N
            if mode == 0 or mode == 3:
                acc += bias[n]
                if relu and acc < 0:
                    acc = float32(0)
            out[index] = acc
    return kernel


GEMM = tuple(_make_gemm(i) for i in range(6))


@cuda.jit(cache=True)
def merge_partials(parts, out, E, P):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = float32(0)
        for p in range(P):
            value += parts[p * E + i]
        out[i] = value


@cuda.jit(cache=True)
def copy_flat(x, y, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        y[i] = x[i]


@cuda.jit(cache=True)
def relu_backward(grad, activation, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if activation[i] <= 0:
            grad[i] = float32(0)


@cuda.jit(cache=True)
def dropout(x, mask, out, factor, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        out[i] = x[i] * float32(mask[i]) * factor


@cuda.jit(cache=True)
def pool_forward(x, y, arg, b, H, W, C):
    E = b * (H // 2) * (W // 2) * C
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        w = (i // C) % (W // 2)
        h = (i // (C * (W // 2))) % (H // 2)
        n = i // (C * (W // 2) * (H // 2))
        base = ((n * H + 2 * h) * W + 2 * w) * C + c
        value, pos = x[base], 0
        for p in range(1, 4):
            candidate = x[base + (p // 2) * W * C + (p % 2) * C]
            if candidate > value:
                value, pos = candidate, p
        y[i], arg[i] = value, pos


@cuda.jit(cache=True)
def pool_backward(dy, arg, dx, b, H, W, C):
    for i in range(cuda.grid(1), b * H * W * C, cuda.gridsize(1)):
        c, w = i % C, (i // C) % W
        h, n = (i // (C * W)) % H, i // (C * W * H)
        out_i = ((n * (H // 2) + h // 2) * (W // 2) + w // 2) * C + c
        dx[i] = float32(0)
        if arg[out_i] == 2 * (h % 2) + w % 2:
            dx[i] = dy[out_i]


@cuda.jit(cache=True, fastmath=False)
def reduce_channels(x, dy, mean, inv, parts1, parts2, L, C, mode):
    # mode0 sum; mode1 centered variance; mode2 BN backward pair.
    s1 = cuda.shared.array((8, 32), float32)
    s2 = cuda.shared.array((8, 32), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c = cuda.blockIdx.y * 32 + tx
    v1, v2 = float32(0), float32(0)
    if c < C:
        for j in range(256):
            l = cuda.blockIdx.x * 2048 + ty + 8 * j
            if l < L:
                value = x[l * C + c]
                if mode == 0:
                    v1 += value
                elif mode == 1:
                    centered = value - mean[c]
                    v1 += centered * centered
                else:
                    g = dy[l * C + c]
                    v1 += g
                    v2 += g * ((value - mean[c]) * inv[c])
    s1[ty, tx], s2[ty, tx] = v1, v2
    cuda.syncthreads()
    stride = 4
    while stride > 0:
        if ty < stride:
            s1[ty, tx] += s1[ty + stride, tx]
            s2[ty, tx] += s2[ty + stride, tx]
        cuda.syncthreads()
        stride //= 2
    if ty == 0 and c < C:
        parts1[cuda.blockIdx.x * C + c] = s1[0, tx]
        parts2[cuda.blockIdx.x * C + c] = s2[0, tx]


@cuda.jit(cache=True, fastmath=False)
def finish_channels(p1, p2, out1, out2, movmean, movvar, mean, inv, P, C, L, mode):
    # mode0 mean; mode1 variance+moving state; mode2 dbeta/dgamma; mode3 bias.
    s1 = cuda.shared.array(256, float32)
    s2 = cuda.shared.array(256, float32)
    t, c = cuda.threadIdx.x, cuda.blockIdx.x
    s1[t], s2[t] = float32(0), float32(0)
    if t < P:
        s1[t], s2[t] = p1[t * C + c], p2[t * C + c]
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s1[t] += s1[t + stride]
            s2[t] += s2[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        if mode == 0:
            out1[c] = s1[0] / float32(L)
        elif mode == 1:
            var = s1[0] / float32(L)
            inv[c] = float32(1) / math.sqrt(var + float32(.001))
            movmean[c] = float32(.9) * movmean[c] + float32(.1) * mean[c]
            movvar[c] = float32(.9) * movvar[c] + float32(.1) * var
        elif mode == 2:
            out1[c], out2[c] = s1[0], s2[0]
        else:
            out1[c] = s1[0]


@cuda.jit(cache=True, fastmath=False)
def bn_forward(x, y, gamma, beta, mean, inv, E, C):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        y[i] = gamma[c] * ((x[i] - mean[c]) * inv[c]) + beta[c]


@cuda.jit(cache=True, fastmath=False)
def bn_backward(x, dy, dx, gamma, mean, inv, dbeta, dgamma, L, C):
    for i in range(cuda.grid(1), L * C, cuda.gridsize(1)):
        c = i % C
        xhat = (x[i] - mean[c]) * inv[c]
        dx[i] = (gamma[c] * inv[c] / float32(L)) * (
            float32(L) * dy[i] - dbeta[c] - xhat * dgamma[c])


@cuda.jit(cache=True, fastmath=False)
def softmax_loss(logits, labels, probs, dlogits, loss, correct, b):
    s = cuda.shared.array(32, float32)
    t, n = cuda.threadIdx.x, cuda.blockIdx.x
    z = float32(-math.inf)
    if t < 29:
        z = logits[n * 29 + t]
    s[t] = z
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] = max(s[t], s[t + stride])
        cuda.syncthreads()
        stride //= 2
    maximum = s[0]
    cuda.syncthreads()
    value = float32(0)
    if t < 29:
        value = math.exp(z - maximum)
    s[t] = value
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
        cuda.syncthreads()
        stride //= 2
    denominator = s[0]
    label = labels[n]
    if t < 29:
        probability = value / denominator
        probs[n * 29 + t] = probability
        dlogits[n * 29 + t] = (probability - float32(t == label)) / float32(b)
    if t == 0:
        loss[n] = (maximum - logits[n * 29 + label]) + math.log(denominator)
        winner = 0
        for c in range(1, 29):
            if logits[n * 29 + c] > logits[n * 29 + winner]:
                winner = c
        correct[n] = winner == label


@cuda.jit(cache=True, fastmath=False)
def adam(theta, grad, moment, velocity, alpha, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        g = grad[i]
        if not math.isfinite(g):
            cuda.atomic.max(error, 0, 1)
        if error[0] == 0:
            m = moment[i] + (g - moment[i]) * float32(.1)
            v = velocity[i] + (g * g - velocity[i]) * float32(.001)
            value = theta[i] - (m * alpha) / (math.sqrt(v) + float32(1e-7))
            if not math.isfinite(value):
                cuda.atomic.max(error, 0, 2)
            theta[i], moment[i], velocity[i] = value, m, v


@cuda.jit(cache=True)
def finite_check(x, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if not math.isfinite(x[i]):
            cuda.atomic.max(error, 0, 3)


@cuda.jit(cache=True)
def metrics(loss, correct, b, total):
    s = cuda.shared.array(256, float32)
    q = cuda.shared.array(256, float32)
    t = cuda.threadIdx.x
    s[t], q[t] = float32(0), float32(0)
    if t < b:
        s[t], q[t] = loss[t], float32(correct[t])
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
            q[t] += q[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        total[0] += s[0]
        total[1] += q[0]
        total[2] += float32(b)


@cuda.jit(cache=True)
def advance_step(counter):
    if cuda.grid(1) == 0:
        counter[0] += 1


@cuda.jit(cache=True)
def im2col(x, cols, R, H, W, Ci):
    K = 9 * Ci
    for i in range(cuda.grid(1), R * K, cuda.gridsize(1)):
        r, kk = i // K, i % K
        n = r // (H * W)
        h = (r // W) % H + kk // (3 * Ci) - 1
        w = r % W + (kk // Ci) % 3 - 1
        value = float32(0)
        if 0 <= h < H and 0 <= w < W:
            value = x[((n * H + h) * W + w) * Ci + kk % Ci]
        cols[i] = value


@cuda.jit(cache=True)
def col2im_gather(cols, dx, R, H, W, Ci):
    for i in range(cuda.grid(1), R * Ci, cuda.gridsize(1)):
        c = i % Ci
        w, h = (i // Ci) % W, (i // (Ci * W)) % H
        n = i // (Ci * W * H)
        value = float32(0)
        for kh in range(3):
            for kw in range(3):
                ho, wo = h - kh + 1, w - kw + 1
                if 0 <= ho < H and 0 <= wo < W:
                    row = (n * H + ho) * W + wo
                    value += cols[row * 9 * Ci + (kh * 3 + kw) * Ci + c]
        dx[i] = value


@cuda.jit(cache=True)
def bias_activation(y, bias, E, C, relu):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = y[i] + bias[i % C]
        if relu and value < 0:
            value = float32(0)
        y[i] = value

```


### model.py

```python
"""Complete CNN-B training on explicitly launched CUDA Python kernels.

TensorFlow is used only as a CPU oracle and for CPU validation/checkpoint I/O.
The original network and its initial state are imported without editing them.
"""
from contextlib import contextmanager
from pathlib import Path
import hashlib
import json
import time
import numpy as np
from .runtime import cuda, device_info
from . import kernels as k

ROOT = Path(__file__).resolve().parents[2]
INITIAL = ROOT / "data/EXP-002/RUN-20261117T171942Z-cpu-01/CNN-B.estado-inicial.npz"
INITIAL_SHA = "10558d723213752b759626c3b196665f2547a0b65cbce296e47b3a4f992305fd"


def cpu_reference():
    """Build original definitions, confirm CPU placement and restore ALL 80 arrays."""
    import os
    for name, value in {"TF_NUM_INTRAOP_THREADS":"6", "TF_NUM_INTEROP_THREADS":"1",
                        "OMP_NUM_THREADS":"6", "OPENBLAS_NUM_THREADS":"6",
                        "MKL_NUM_THREADS":"6", "TF_DETERMINISTIC_OPS":"1",
                        "TF_ENABLE_ONEDNN_OPTS":"1"}.items():
        os.environ[name] = value
    import tensorflow as tf
    tf.config.set_visible_devices([], "GPU")
    tf.config.threading.set_intra_op_parallelism_threads(6)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.utils.set_random_seed(43)
    tf.keras.mixed_precision.set_global_policy("float32")
    import sys
    sys.path.insert(0, str(ROOT / "scripts"))
    from benchmark_cpu import original_namespace
    ns = original_namespace()
    ns.update(keras=tf.keras, layers=tf.keras.layers, NUM_CLASSES=29)
    model = ns["build_cnn_b"]()
    model.jit_compile = False
    model.optimizer.build(model.trainable_variables)
    if hashlib.sha256(INITIAL.read_bytes()).hexdigest() != INITIAL_SHA:
        raise ValueError("Initial checkpoint identity changed")
    with np.load(INITIAL, allow_pickle=False) as archive:
        variables = list(model.variables) + list(model.optimizer.variables)
        if len(variables) != len(archive.files):
            raise ValueError("Initial state variable count differs")
        for index, variable in enumerate(variables):
            value = archive[f"arr_{index}"]
            if tuple(variable.shape) != value.shape or str(variable.dtype) != str(value.dtype):
                raise ValueError(f"Incompatible state variable {index}")
            variable.assign(value)
    if int(model.optimizer.iterations.numpy()) != 0:
        raise ValueError("Expected optimizer iteration zero")
    if any(np.any(v.numpy()) for v in [*model.optimizer._momentums, *model.optimizer._velocities]):
        raise ValueError("Nonzero initial Adam requires explicit state mapping")
    return tf, model


def grid(E):
    return max(1, min(4096, (int(E) + 255) // 256))


def compiled_resources():
    """Numba-CUDA 0.30 API expects signature objects, not .signatures tuples."""
    result={}
    for name,kern in vars(k).items():
        if hasattr(kern,"get_regs_per_thread") and kern.signatures:
            regs=kern.get_regs_per_thread()
            shared=kern.get_shared_mem_per_block()
            local=kern.get_local_mem_per_thread()
            result[name]=[dict(firma=str(sig),registros=value,shared_bytes=shared[sig],local_bytes=local[sig])
                          for sig,value in regs.items()]
    return result


class Network:
    def __init__(self, reference, backend="explicit"):
        self.info = device_info()
        if self.info["free_bytes"] < 1536 * 2**20:
            raise MemoryError("Insufficient free VRAM for the frozen profile")
        self.stream = cuda.stream()
        if backend not in ("explicit", "cublas"):
            raise ValueError("Unknown backend")
        self.backend = backend
        self.allocations, self.params, self.ops, self.transfers = [], [], [], []
        self.reference = reference
        self.input = self.alloc(128 * 64 * 64 * 3)
        self.host_x = cuda.pinned_array(128 * 64 * 64 * 3, dtype=np.float32)
        self.labels = self.alloc(128, np.int64)
        self.host_y = cuda.pinned_array(128, dtype=np.int64)
        shape, previous = (64, 64, 3), self.input
        for layer in reference.layers:
            name = type(layer).__name__
            item = dict(input=previous, shape=shape, layer=layer)
            if name == "Conv2D":
                H, W, Ci = shape
                Co = layer.filters
                item.update(kind="conv", H=H, W=W, Ci=Ci, Co=Co,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias), relu=True)
                shape = (H, W, Co)
            elif name == "Dense":
                I, O = int(layer.kernel.shape[0]), int(layer.kernel.shape[1])
                item.update(kind="dense", I=I, O=O, relu=O != 29,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias))
                shape = (O,)
            elif name == "BatchNormalization":
                C = shape[-1]
                item.update(kind="bn", C=C, gamma=self.param(layer.gamma), beta=self.param(layer.beta),
                            moving_mean=self.alloc(C), moving_var=self.alloc(C),
                            initial_mean=layer.moving_mean.numpy().copy(), initial_var=layer.moving_variance.numpy().copy(),
                            mean=self.alloc(C), inv=self.alloc(C))
            elif name == "MaxPooling2D":
                H, W, C = shape
                item.update(kind="pool", H=H, W=W, C=C)
                shape = (H // 2, W // 2, C)
                item["arg"] = self.alloc(128 * int(np.prod(shape)), np.uint8)
            elif name == "Dropout":
                E = 128 * int(np.prod(shape))
                item.update(kind="drop", p=layer.rate, mask=self.alloc(E, np.uint8),
                            host_mask=cuda.pinned_array(E, dtype=np.uint8))
            elif name == "Flatten":
                shape = (int(np.prod(shape)),)
                continue
            elif name == "InputLayer":
                continue
            else:
                raise TypeError(f"Unsupported original layer {name}")
            item["sample_elements"] = int(np.prod(shape))
            item["output"] = self.alloc(128 * item["sample_elements"])
            self.ops.append(item)
            previous = item["output"]
        self.probs = self.alloc(128 * 29)
        self.loss, self.correct = self.alloc(128), self.alloc(128, np.uint8)
        self.g1, self.g2 = self.alloc(128 * 64 * 64 * 32), self.alloc(128 * 64 * 64 * 32)
        self.parts = self.alloc(16 * 2**20 // 4)
        self.p1, self.p2 = self.alloc(65536), self.alloc(65536)
        self.dummy = self.alloc(256)
        self.total = self.alloc(3)
        self.error = self.alloc(1, np.int32)
        self.counter = self.alloc(1, np.int64)
        self.host_error = cuda.pinned_array(1, dtype=np.int32)
        self.host_total = cuda.pinned_array(3, dtype=np.float32)
        if backend == "cublas":
            from .blas import Blas
            self.blas = Blas(self.stream)
            self.columns = self.alloc(128 * 64 * 64 * (9 * 32))
        self.bytes_allocated = sum(a.nbytes for a in self.allocations)
        if self.bytes_allocated > (1408 if backend == "cublas" else 768) * 2**20:
            raise MemoryError(f"Arena exceeds design: {self.bytes_allocated}")
        if sum(p["value"].size for p in self.params) != 2245245:
            raise ValueError("Architecture parameter count changed")
        self.events = []
        self.restore()

    def alloc(self, count, dtype=np.float32):
        array = cuda.device_array(int(count), dtype=dtype, stream=self.stream)
        self.allocations.append(array)
        return array

    def param(self, variable):
        values = variable.numpy().copy()
        p = dict(initial=values, value=self.alloc(values.size), grad=self.alloc(values.size),
                 m=self.alloc(values.size), v=self.alloc(values.size), variable=variable)
        self.params.append(p)
        return p

    def restore(self):
        for p in self.params:
            p["value"].copy_to_device(p["initial"].ravel(), stream=self.stream)
            for state in ("m", "v"):
                p[state].copy_to_device(np.zeros(p["initial"].size, np.float32), stream=self.stream)
        for op in self.ops:
            if op["kind"] == "bn":
                op["moving_mean"].copy_to_device(op["initial_mean"], stream=self.stream)
                op["moving_var"].copy_to_device(op["initial_var"], stream=self.stream)
        for a in (self.error, self.counter, self.total):
            a.copy_to_device(np.zeros(a.size, dtype=a.dtype), stream=self.stream)
        self.stream.synchronize()
        self.steps = 0
        self.events = []

    @contextmanager
    def timed(self, phase):
        start, end = cuda.event(timing=True), cuda.event(timing=True)
        start.record(self.stream)
        yield
        end.record(self.stream)
        self.events.append((phase, start, end))

    def drain_events(self):
        self.stream.synchronize()
        result = {}
        for phase, start, end in self.events:
            result[phase] = result.get(phase, 0.) + cuda.event_elapsed_time(start, end) / 1000.
        self.events.clear()
        return result

    def gemm(self, mode, a, v, bias, out, M, N, K, H=1, W=1, Ci=1, Co=1, I=1, O=1, relu=False):
        if self.backend == "cublas":
            if mode in (0, 2):
                R = M if mode == 0 else K
                width = 9 * Ci
                k.im2col[grid(R * width), 256, self.stream](a, self.columns, R, H, W, Ci)
                self.blas.gemm(self.columns, v, out, M, N, K, transpose_a=mode == 2)
            elif mode == 1:
                self.blas.gemm(a, v, self.columns, M, 9 * Ci, Co, transpose_b=True)
                k.col2im_gather[grid(M * Ci), 256, self.stream](self.columns, out, M, H, W, Ci)
            else:
                self.blas.gemm(a, v, out, M, N, K, transpose_a=mode == 5, transpose_b=mode == 4)
            if mode in (0, 3):
                k.bias_activation[grid(M * N), 256, self.stream](out, bias, M * N, N, relu)
            return
        P = (K + 2047) // 2048 if mode in (2, 5) else 1
        if mode in (2, 5) and P * M * N > self.parts.size:
            raise MemoryError("Partial gradient workspace exhausted")
        for m0 in range(0, M, 16384):
            geom = ((N + 15) // 16, (min(16384, M - m0) + 15) // 16, P)
            k.GEMM[mode][geom, (16, 16), self.stream](
                a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0)

    def reduce(self, x, dy, op, L, C, mode, out1, out2):
        P = (L + 2047) // 2048
        if P > 256 or P * C > self.p1.size:
            raise ValueError("Reduction shape exceeds frozen profile")
        mean, inv = op.get("mean", self.dummy), op.get("inv", self.dummy)
        k.reduce_channels[(P, (C + 31) // 32), (32, 8), self.stream](
            x, dy, mean, inv, self.p1, self.p2, L, C, 2 if mode == 2 else (1 if mode == 1 else 0))
        k.finish_channels[C, 256, self.stream](self.p1, self.p2, out1, out2,
            op.get("moving_mean", self.dummy), op.get("moving_var", self.dummy),
            mean, inv, P, C, L, mode)

    def upload(self, x, labels, epoch, batch_index):
        b = len(x)
        np.copyto(self.host_x[:x.size], x.ravel())
        np.copyto(self.host_y[:b], labels)
        drops = [op for op in self.ops if op["kind"] == "drop"]
        for d, op in enumerate(drops):
            E = b * op["sample_elements"]
            rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([43, epoch, batch_index, d + 1])))
            op["host_mask"][:E] = rng.random(E, dtype=np.float32) >= np.float32(op["p"])
        with self.timed("h2d_lotes"):
            self.input[:x.size].copy_to_device(self.host_x[:x.size], stream=self.stream)
            self.labels[:b].copy_to_device(self.host_y[:b], stream=self.stream)
            for op in drops:
                E = b * op["sample_elements"]
                op["mask"][:E].copy_to_device(op["host_mask"][:E], stream=self.stream)
        self.transfers.append(dict(batch=b, imagenes_bytes=x.nbytes, etiquetas_bytes=labels.nbytes,
                                   mascaras_bytes=sum(b * op["sample_elements"] for op in drops)))

    def forward(self, b):
        for op in self.ops:
            kind, x, y, E = op["kind"], op["input"], op["output"], b * op["sample_elements"]
            if kind == "conv":
                H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                self.gemm(0, x, op["weights"]["value"], op["bias"]["value"], y,
                          b * H * W, Co, 9 * Ci, H, W, Ci, Co, relu=True)
            elif kind == "dense":
                self.gemm(3, x, op["weights"]["value"], op["bias"]["value"], y,
                          b, op["O"], op["I"], I=op["I"], O=op["O"], relu=op["relu"])
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, self.dummy, op, L, C, 0, op["mean"], self.dummy)
                self.reduce(x, self.dummy, op, L, C, 1, self.dummy, self.dummy)
                k.bn_forward[grid(E), 256, self.stream](x, y, op["gamma"]["value"],
                    op["beta"]["value"], op["mean"], op["inv"], E, C)
            elif kind == "pool":
                k.pool_forward[grid(E), 256, self.stream](x, y, op["arg"], b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](x, op["mask"], y, np.float32(1 / (1 - op["p"])), E)
            k.finite_check[grid(E), 256, self.stream](y, E, self.error)
        k.softmax_loss[b, 32, self.stream](self.ops[-1]["output"], self.labels,
            self.probs, self.g1, self.loss, self.correct, b)
        k.finite_check[grid(b), 256, self.stream](self.loss, b, self.error)

    def backward(self, b, audit_hook=None):
        current, other = self.g1, self.g2
        for index in range(len(self.ops) - 1, -1, -1):
            op = self.ops[index]
            kind, x = op["kind"], op["input"]
            E = b * op["sample_elements"]
            if kind in ("conv", "dense"):
                if op["relu"]:
                    k.relu_backward[grid(E), 256, self.stream](current, op["output"], E)
                if kind == "conv":
                    H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                    L = b * H * W
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(2, x, current, self.dummy, target, 9 * Ci, Co, L, H, W, Ci, Co)
                    if self.backend == "explicit":
                        k.merge_partials[grid(9 * Ci * Co), 256, self.stream](self.parts,
                            op["weights"]["grad"], 9 * Ci * Co, (L + 2047) // 2048)
                    if index:
                        self.gemm(1, current, op["weights"]["value"], self.dummy, other,
                                  L, Ci, 9 * Co, H, W, Ci, Co)
                    C = Co
                else:
                    I, O, L, C = op["I"], op["O"], b, op["O"]
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(5, x, current, self.dummy, target, I, O, b, I=I, O=O)
                    if self.backend == "explicit":
                        k.merge_partials[grid(I * O), 256, self.stream](self.parts, op["weights"]["grad"], I * O, 1)
                    self.gemm(4, current, op["weights"]["value"], self.dummy, other, b, I, O, I=I, O=O)
                self.reduce(current, self.dummy, {}, L, C, 3, op["bias"]["grad"], self.dummy)
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, current, op, L, C, 2, op["beta"]["grad"], op["gamma"]["grad"])
                k.bn_backward[grid(E), 256, self.stream](x, current, other, op["gamma"]["value"],
                    op["mean"], op["inv"], op["beta"]["grad"], op["gamma"]["grad"], L, C)
            elif kind == "pool":
                E_in = b * op["H"] * op["W"] * op["C"]
                k.pool_backward[grid(E_in), 256, self.stream](current, op["arg"], other,
                    b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](current, op["mask"], other,
                    np.float32(1 / (1 - op["p"])), E)
            if audit_hook is not None:
                self.stream.synchronize()
                audit_hook(index, op, current, other)
            current, other = other, current

    def update(self, b):
        t = np.float32(self.steps + 1)
        alpha = np.float32(.001) * np.sqrt(np.float32(1) - np.float32(.999)**t) / (
            np.float32(1) - np.float32(.9)**t)
        for p in self.params:
            k.adam[grid(p["value"].size), 256, self.stream](p["value"], p["grad"],
                p["m"], p["v"], np.float32(alpha), p["value"].size, self.error)
        k.metrics[1, 256, self.stream](self.loss, self.correct, b, self.total)
        k.advance_step[1, 1, self.stream](self.counter)
        self.steps += 1

    def train_step(self, x, labels, epoch=0, batch_index=0):
        self.upload(x, labels, epoch, batch_index)
        with self.timed("forward_loss"):
            self.forward(len(x))
        with self.timed("backward"):
            self.backward(len(x))
        with self.timed("adam_metricas"):
            self.update(len(x))
        with self.timed("d2h_control"):
            self.error.copy_to_host(self.host_error, stream=self.stream)
        times = self.drain_events()
        if self.host_error[0]:
            raise FloatingPointError(f"CUDA sticky error {self.host_error[0]}, step {self.steps}")
        return times

    def reset_metrics(self):
        self.total.copy_to_device(np.zeros(3, np.float32), stream=self.stream)

    def export(self, optimizer=False):
        """D2H model state, then install CPU mirror outside the transfer interval."""
        host = []
        with self.timed("d2h_modelo"):
            for p in self.params:
                host.append((p["variable"], p["value"].copy_to_host(stream=self.stream).reshape(p["initial"].shape)))
            for op in self.ops:
                if op["kind"] == "bn":
                    host.append((op["layer"].moving_mean, op["moving_mean"].copy_to_host(stream=self.stream)))
                    host.append((op["layer"].moving_variance, op["moving_var"].copy_to_host(stream=self.stream)))
            self.total.copy_to_host(self.host_total, stream=self.stream)
        times = self.drain_events()
        for variable, value in host:
            variable.assign(value)
        if optimizer:
            with self.timed("d2h_optimizador"):
                states = [(p["m"].copy_to_host(stream=self.stream), p["v"].copy_to_host(stream=self.stream)) for p in self.params]
                count = self.counter.copy_to_host(stream=self.stream)
            extra = self.drain_events()
            times.update(extra)
            if int(count[0]) != self.steps:
                raise ValueError("Device update counter differs")
            self.reference.optimizer.iterations.assign(int(count[0]))
            for i, (moment, velocity) in enumerate(states):
                self.reference.optimizer._momentums[i].assign(moment.reshape(self.params[i]["initial"].shape))
                self.reference.optimizer._velocities[i].assign(velocity.reshape(self.params[i]["initial"].shape))
        return times, self.host_total.copy()

```


### runtime.py

```python
"""Local toolkit discovery. Never installs packages or silently falls back to CPU."""
import os
from pathlib import Path

_DLL_HANDLES = []


def configure():
    if os.environ.get("NUMBA_ENABLE_CUDASIM", "0") != "0":
        raise RuntimeError("CUDA simulator is not a GPU benchmark.")
    if os.name == "nt":
        root = Path(os.environ.get("CUDA_HOME", "C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v13.1"))
        os.environ.setdefault("CUDA_HOME", str(root))
        for folder in (root / "bin/x64", root / "bin", root / "nvvm/bin/x64", root / "nvvm/bin"):
            if folder.is_dir():
                _DLL_HANDLES.append(os.add_dll_directory(str(folder)))


configure()
# numba-cuda 0.30.4 still registers NumPy's retired row_stack alias at import.
# Preserve the installed NumPy 2.5.5: the alias is exactly vstack, not new math.
# Process-local compatibility only; no site-packages files are changed.
import numpy as np
if not hasattr(np, "row_stack"):
    np.row_stack = np.vstack
from numba import cuda  # noqa: E402


def device_info():
    if not cuda.is_available():
        raise RuntimeError("A real CUDA device is required.")
    d = cuda.get_current_device()
    free, total = cuda.current_context().get_memory_info()
    return dict(name=d.name.decode() if isinstance(d.name, bytes) else d.name,
                compute_capability=list(d.compute_capability), free_bytes=int(free), total_bytes=int(total),
                max_threads_per_block=int(d.MAX_THREADS_PER_BLOCK),
                max_shared_memory_per_block=int(d.MAX_SHARED_MEMORY_PER_BLOCK),
                simulator=False)

```
